# Maya-CSM TTS Server (Kaggle)

Runs the Maya TTS API on a free Kaggle GPU and exposes it via a tunnel (ngrok or Cloudflare) for SillyTavern.

**Before running (all required):**
1. Settings panel (right sidebar) -> Accelerator -> **GPU T4 x2**. (**Not P100** - Kaggle's PyTorch is 2.10+cu128, which dropped Pascal/sm_60; the model won't load on a P100.)
2. Settings panel -> **Internet -> On** (requires a phone-verified Kaggle account).
3. Accept the model terms at https://huggingface.co/sesame/csm-1b with your HF account.
4. Add-ons -> Secrets -> add `HF_TOKEN` (Hugging Face read token) and attach it. *Optional:* add `NGROK_AUTH_TOKEN` (from https://dashboard.ngrok.com) for a stable tunnel URL.
5. Set `REPO_URL` below to your fork/copy of the maya-csm repo.

**Performance** - full detail in `docs/PERFORMANCE-IMPROVEMENTS.md`:

- **T4 x2 is the default and the only working Kaggle accelerator.** A reply's sentences are split into chunks and generated one per T4, so multi-sentence replies come back ~2x faster; a one-chunk reply uses a single T4.
- **`MAYA_COMPILE=1`** (env cell, commented out) is `torch.compile` + CUDA graphs on one T4 with greedy decoding - it removes launch overhead from the 31-step depth-decoder loop (big for short replies) but compiles for ~1-3 min and PyTorch 2.10 has a reported `reduce-overhead` regression. **Try it, then compare with the benchmark cell** - don't assume it's faster.
- **In SillyTavern:** enable **"Narrate by paragraphs (when not streaming)"** and keep the first sentence short - paragraph 1 plays while the rest generate. Biggest *perceived* win, costs nothing. See `docs/SILLYTAVERN.md`.
- `MAYA_DTYPE` defaults to `float16` (right for T4); `MAYA_DEVICES` overrides the device list.

In [ ]:
REPO_URL = "https://github.com/l0ophole/maya-csm"  # <-- change me

!git clone -q $REPO_URL /kaggle/working/maya-csm
%pip install -q "/kaggle/working/maya-csm[model]"
# Kaggle preinstalls may predate CSM support in transformers
%pip install -q -U transformers peft accelerate torchao
%pip install -q pyngrok  # optional ngrok tunnel (see the tunnel cells below)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["MAYA_ADAPTER"] = "shb777/csm-maya-exp2"  # pull LoRA from the Hub
os.environ["MAYA_PRELOAD"] = "1"   # load the model at startup, not lazily on the first request

# Default: both T4s, data-parallel across a reply's chunks.
#
# Alternative - torch.compile + CUDA graphs on ONE T4, greedy decoding. Faster per
# stream for short replies IN THEORY, but ~1-3 min compile on the warm-up cell and a
# reported PyTorch 2.10 reduce-overhead regression - benchmark it before adopting.
# os.environ["MAYA_COMPILE"] = "1"
#
# os.environ["MAYA_DTYPE"] = "bfloat16"  # fall back from the float16 default if you hear artifacts
# os.environ["MAYA_DEVICES"] = "cuda:0"  # pin to one GPU (e.g. to A/B against the 2-GPU path)

In [ ]:
# Start the server in a background thread. With MAYA_PRELOAD=1 this cell blocks
# while the model downloads + loads + warms up - several minutes on the first run.
import threading
import uvicorn
from maya_csm.config import Settings
from maya_csm.server import create_app

app = create_app(Settings.from_env())
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"),
    daemon=True,
).start()
print("server starting on :8000")

In [ ]:
# Warm-up & first timing. MAYA_PRELOAD=1 already loaded + warmed the model in the
# previous cell (cuDNN autotune, allocator), so both calls below should be fast and
# similar. If you set MAYA_COMPILE=1, the first call here also triggers the ~1-3 min
# torch.compile capture. Re-run this cell after restarting the server.
import time
import requests

for _ in range(90):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)


def _timed(text):
    t0 = time.perf_counter()
    r = requests.post("http://localhost:8000/v1/audio/speech",
                      json={"input": text, "voice": "maya"}, timeout=600)
    r.raise_for_status()
    return time.perf_counter() - t0


print(f"  first call:  {_timed('Hey there, warming up my voice.'):5.1f}s")
print(f"  second call: {_timed('This one should be quick now that everything is warm.'):5.1f}s  <- steady state")

In [ ]:
# GPU check - run during a multi-sentence request too: on T4 x2 both rows should show
# memory resident, and utilization should spike on both while a split reply generates.
!nvidia-smi --query-gpu=index,name,memory.used,utilization.gpu --format=csv

### Tunnel - option A: ngrok (stable URL)

A stable URL you set once in SillyTavern, instead of re-pasting a new one each session.
Add an `NGROK_AUTH_TOKEN` secret (Add-ons -> Secrets, token from https://dashboard.ngrok.com).
If the secret is missing this cell no-ops and the Cloudflare quick tunnel (option B) is used.

In [ ]:
PUBLIC_URL = None
try:
    _ngrok_token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
except Exception:
    _ngrok_token = None

if _ngrok_token:
    from pyngrok import conf, ngrok

    conf.get_default().auth_token = _ngrok_token
    ngrok.kill()  # drop stale tunnels from a previous run
    PUBLIC_URL = ngrok.connect(8000, "http").public_url.replace("http://", "https://")
    print("ngrok tunnel:", PUBLIC_URL)
    print(f"\nSillyTavern OpenAI-Compatible TTS endpoint:\n  {PUBLIC_URL}/v1/audio/speech")
else:
    print("No NGROK_AUTH_TOKEN secret - skipping ngrok; use the Cloudflare tunnel below.")

In [ ]:
# Tunnel - option B: Cloudflare quick tunnel (no account; new random URL every run).
# Auto-skipped if ngrok (option A) already produced a URL. Re-run if the URL dies.
import re
import subprocess
import time

try:
    PUBLIC_URL
except NameError:
    PUBLIC_URL = None

if PUBLIC_URL:
    print("Using the ngrok URL from option A:", PUBLIC_URL)
else:
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared

    proc = subprocess.Popen(
        ["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    deadline = time.time() + 30
    while time.time() < deadline and PUBLIC_URL is None:
        line = proc.stdout.readline()
        m = re.search(r"https://[\w.-]+\.trycloudflare\.com", line)
        if m:
            PUBLIC_URL = m.group(0)
    print("Cloudflare tunnel:", PUBLIC_URL)
    print(f"\nSillyTavern OpenAI-Compatible TTS endpoint:\n  {PUBLIC_URL}/v1/audio/speech")

In [ ]:
# In-notebook sanity check (no tunnel needed) - listen to a sample
import requests
from IPython.display import Audio

r = requests.post(
    "http://localhost:8000/v1/audio/speech",
    json={"input": "[giggling] Hey there, it's so good to hear your voice.", "voice": "maya"},
)
r.raise_for_status()
Audio(r.content)

### Benchmark (optional but recommended)

Round-trip time for 1 / 2 / 4-sentence requests against the local server (no tunnel), plus the
real-time factor (RTF = generate time / audio seconds; RTF < 1 means generation outruns playback).

Run it once with the defaults, then again after uncommenting the `os.environ["MAYA_COMPILE"] = "1"`
line in the env cell and restarting the runtime. Keep whichever is faster for your typical reply
length - PyTorch 2.10 has a reported `torch.compile` `reduce-overhead` regression, so compile is
*not* a safe assumption.

In [ ]:
import io
import time

import requests
import soundfile as sf

_SENT = "This is a fairly ordinary sentence of the kind a character might say."
for n in (1, 2, 4):
    text = " ".join([_SENT] * n)
    t0 = time.perf_counter()
    r = requests.post("http://localhost:8000/v1/audio/speech",
                      json={"input": text, "voice": "maya"}, timeout=600)
    r.raise_for_status()
    wall = time.perf_counter() - t0
    audio, sr = sf.read(io.BytesIO(r.content))
    secs = len(audio) / sr
    print(f"{n} sentence(s) / {len(text):3d} chars: {wall:5.1f}s wall, "
          f"{secs:4.1f}s audio, RTF={wall / max(secs, 1e-3):.2f}")